# VAZHI DAPT Data Prep v1.1 — Clean Tamil Corpus

**Fixes from v1.0 (incorporating multi-agent review + GPT5.2 feedback):**
1. NFKC Unicode normalization (was missing entirely)
2. Strip `\ufffd`, zero-width chars, control characters
3. Tamil threshold raised from 50% to 70% (50% let in too much noise)
4. Three Sangraha sources: verified + unverified + synthetic (was verified only)
5. Target 50M tokens (was 30M — insufficient for 0.6B model)
6. Whitespace collapse and normalization

```
Step 1 (THIS NOTEBOOK): Clean Data Prep — CPU only
  → Output: CryptoYogi/vazhi-dapt-tamil-v1_1

Step 2: DAPT Training — Kaggle T4 GPU
  → Input:  This dataset + Qwen3-0.6B (instruct, NOT base)
  → Output: CryptoYogi/qwen3-0.6b-tamil-v1_1

Step 3: SFT (future)
```

**Runtime:** ~30-60 min on CPU (streaming + tokenization)

In [1]:
!pip install -q -U \
  "transformers>=4.45.0,<5.0.0" \
  "datasets>=2.21.0" \
  "huggingface_hub>=0.24.7"

print("\u2705 Dependencies installed (CPU-only)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.5 MB/s eta 0:00:00
✅ Dependencies installed (CPU-only)


In [2]:
import os
import re
import json
import random
import hashlib
import unicodedata
import numpy as np
from collections import Counter
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer
from huggingface_hub import login, HfApi

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === KEY CONFIG ===
BASE_MODEL = "Qwen/Qwen3-0.6B"  # Instruct model tokenizer (matches training)
OUTPUT_DATASET = "CryptoYogi/vazhi-dapt-tamil-v1_1"

# === TOKEN BUDGET ===
TARGET_TOKENS = 50_000_000     # 50M tokens (was 30M — insufficient)
MAX_SEQ_LENGTH = 1024

# === DATA QUALITY FILTERS (stricter than v1.0) ===
MIN_TAMIL_PCT = 70             # Raised from 50% — eliminates noisy mixed-language docs
MIN_DOC_CHARS = 200
MAX_DOC_CHARS = 8000
MAX_REPETITION_RATIO = 0.5

# === EVAL SPLIT ===
EVAL_PCT = 0.02

# === SANGRAHA SOURCES (priority order) ===
SANGRAHA_SOURCES = [
    {"config": "verified",  "split": "tam",      "label": "verified"},
    {"config": "unverified", "split": "tam",     "label": "unverified"},
    {"config": "synthetic", "split": "tam_Taml", "label": "synthetic"},
]

print(f"\U0001f4cb DAPT Data Prep v1.1 Config:")
print(f"   Tokenizer:    {BASE_MODEL} (instruct — matches training)")
print(f"   Output:       {OUTPUT_DATASET}")
print(f"   Token budget: {TARGET_TOKENS:,}")
print(f"   Block size:   {MAX_SEQ_LENGTH} tokens")
print(f"   Tamil >= {MIN_TAMIL_PCT}% (raised from 50%)")
print(f"   Sources:      {[s['label'] for s in SANGRAHA_SOURCES]}")
print(f"   Eval holdout: {EVAL_PCT:.0%}")

📋 DAPT Data Prep v1.1 Config:
   Tokenizer:    Qwen/Qwen3-0.6B (instruct — matches training)
   Output:       CryptoYogi/vazhi-dapt-tamil-v1_1
   Token budget: 50,000,000
   Block size:   1024 tokens
   Tamil >= 70% (raised from 50%)
   Sources:      ['verified', 'unverified', 'synthetic']
   Eval holdout: 2%


In [3]:
# HuggingFace login
# Kaggle: from kaggle_secrets import UserSecretsClient; login(token=UserSecretsClient().get_secret("HF_TOKEN"))
# Colab/local: login()
login()
print("\u2705 Logged in to HuggingFace")

✅ Logged in to HuggingFace


In [4]:
print(f"\U0001f4e5 Loading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"\u2705 Tokenizer ready: {len(tokenizer)} tokens")
print(f"   eos_token: {tokenizer.eos_token!r} (ID {tokenizer.eos_token_id})")

📥 Loading tokenizer from Qwen/Qwen3-0.6B...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Tokenizer ready: 151669 tokens
   eos_token: '<|im_end|>' (ID 151645)


## Text Cleaning Pipeline (NEW in v1.1)

Addresses GPT5.2 finding: DAPT v1.0 had `\ufffd` in decoded training samples.
Root cause: no Unicode normalization + packing boundary artifacts.

In [5]:
# === TEXT CLEANING (NEW — was completely missing in v1.0) ===

# Zero-width and invisible characters to strip
_INVISIBLE_RE = re.compile(
    '['
    '\u200b'  # zero-width space
    '\u200c'  # zero-width non-joiner
    '\u200d'  # zero-width joiner
    '\u200e'  # left-to-right mark
    '\u200f'  # right-to-left mark
    '\u00ad'  # soft hyphen
    '\ufeff'  # byte order mark
    '\u2060'  # word joiner
    '\u2061'  # function application
    '\u2062'  # invisible times
    '\u2063'  # invisible separator
    '\u2064'  # invisible plus
    '\ufff9'  # interlinear annotation anchor
    '\ufffa'  # interlinear annotation separator
    '\ufffb'  # interlinear annotation terminator
    ']'
)

# Control characters (keep newline \n, tab \t, carriage return \r)
_CONTROL_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]')


def clean_text(text):
    """Clean and normalize text for DAPT training.

    Fixes identified by multi-agent review + GPT5.2:
    1. NFKC normalization (combines decomposed Tamil chars)
    2. Remove replacement character (source of decoded gibberish)
    3. Strip zero-width/invisible chars
    4. Remove control characters
    5. Normalize whitespace
    """
    # 1. NFKC Unicode normalization
    text = unicodedata.normalize('NFKC', text)

    # 2. Remove replacement character
    text = text.replace('\ufffd', '')

    # 3. Strip zero-width and invisible characters
    text = _INVISIBLE_RE.sub('', text)

    # 4. Remove control characters (keep \n, \t)
    text = _CONTROL_RE.sub('', text)

    # 5. Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)       # collapse horizontal whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)    # max 2 consecutive newlines
    text = re.sub(r' *\n *', '\n', text)       # strip spaces around newlines

    # 6. Strip leading/trailing
    text = text.strip()

    return text


def count_tamil_chars(text):
    """Count Tamil Unicode characters (U+0B80 to U+0BFF)."""
    return sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')


def tamil_char_pct(text):
    """Tamil character percentage of total text length."""
    if not text:
        return 0.0
    return 100.0 * count_tamil_chars(text) / len(text)


def has_excessive_repetition(text, threshold=MAX_REPETITION_RATIO):
    """Check if a doc has too many repeated lines (boilerplate)."""
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) < 3:
        return False
    line_counts = Counter(lines)
    most_common_count = line_counts.most_common(1)[0][1]
    return most_common_count / len(lines) > threshold


def text_hash(text):
    """Fast MD5 hash for dedup."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()


# Quick test
test_text = "  \u200bதமிழ்\ufffd  நாடு  \n\n\n\n  test  "
cleaned = clean_text(test_text)
print(f"\u2705 Cleaning pipeline ready")
print(f"   Test: {test_text!r}")
print(f"   Clean: {cleaned!r}")
assert '\ufffd' not in cleaned, "Replacement char not removed!"
assert '\u200b' not in cleaned, "Zero-width char not removed!"
assert '\n\n\n' not in cleaned, "Excessive newlines not collapsed!"
print(f"   All assertions passed")

✅ Cleaning pipeline ready
   Test: '  \u200bதமிழ்�  நாடு  \n\n\n\n  test  '
   Clean: 'தமிழ் நாடு\n\ntest'
   All assertions passed


## Stream, Clean, Filter & Collect from All Sangraha Sources

Priority order: verified (highest quality) → unverified (more volume) → synthetic (clean grammar).
Stops when 50M token budget is met.

In [6]:
clean_texts = []
seen_hashes = set()
total_tokens = 0
source_stats = {}

BUFFER_FACTOR = 1.1
effective_budget = int(TARGET_TOKENS * BUFFER_FACTOR)

for src in SANGRAHA_SOURCES:
    if total_tokens >= effective_budget:
        break

    label = src['label']
    print(f"\n\U0001f4e5 Streaming Sangraha {src['config']}/{src['split']} ({label})...")

    stats = {
        "total_seen": 0, "dropped_short": 0, "dropped_long": 0,
        "dropped_tamil": 0, "dropped_repetition": 0, "dropped_dedup": 0,
        "dropped_empty_after_clean": 0, "kept": 0, "tokens": 0,
    }

    try:
        ds_stream = load_dataset(
            "ai4bharat/sangraha", src['config'],
            split=src['split'], streaming=True
        )
    except Exception as e:
        print(f"   \u26a0\ufe0f  Failed to load {label}: {e}")
        source_stats[label] = stats
        continue

    for item in ds_stream:
        if total_tokens >= effective_budget:
            break

        stats["total_seen"] += 1
        raw_text = item.get("text", "")

        # === CLEAN FIRST (NEW in v1.1) ===
        text = clean_text(raw_text)

        if len(text) < MIN_DOC_CHARS:
            stats["dropped_short"] += 1
            continue
        if len(text) > MAX_DOC_CHARS:
            stats["dropped_long"] += 1
            continue

        # Tamil character ratio (on CLEANED text)
        t_pct = tamil_char_pct(text)
        if t_pct < MIN_TAMIL_PCT:
            stats["dropped_tamil"] += 1
            continue

        if has_excessive_repetition(text):
            stats["dropped_repetition"] += 1
            continue

        h = text_hash(text)
        if h in seen_hashes:
            stats["dropped_dedup"] += 1
            continue
        seen_hashes.add(h)

        n_tokens = len(tokenizer.encode(text, add_special_tokens=False))

        clean_texts.append(text)
        total_tokens += n_tokens
        stats["kept"] += 1
        stats["tokens"] += n_tokens

        if stats["kept"] % 2000 == 0:
            pct = 100 * total_tokens / TARGET_TOKENS
            print(f"   ...{label}: {stats['kept']:,} docs, {stats['tokens']:,} tokens (total: {pct:.0f}% of budget)")

    source_stats[label] = stats
    print(f"   \u2705 {label}: kept {stats['kept']:,} / scanned {stats['total_seen']:,} ({stats['tokens']:,} tokens)")

print(f"\n{'='*60}")
print(f"\U0001f4ca FILTERING SUMMARY")
print(f"{'='*60}")
print(f"{'Source':<12} {'Scanned':>8} {'Kept':>8} {'Tokens':>12} {'Keep%':>6}")
print(f"{'-'*48}")
for label, s in source_stats.items():
    keep_pct = 100 * s['kept'] / max(s['total_seen'], 1)
    print(f"{label:<12} {s['total_seen']:>8,} {s['kept']:>8,} {s['tokens']:>12,} {keep_pct:>5.0f}%")
print(f"{'-'*48}")
total_scanned = sum(s['total_seen'] for s in source_stats.values())
total_kept = sum(s['kept'] for s in source_stats.values())
print(f"{'TOTAL':<12} {total_scanned:>8,} {total_kept:>8,} {total_tokens:>12,}")
print(f"\n   Total clean docs:   {len(clean_texts):,}")
print(f"   Total clean tokens: {total_tokens:,}")
print(f"   Budget coverage:    {100 * total_tokens / TARGET_TOKENS:.0f}%")

if total_tokens < TARGET_TOKENS * 0.9:
    print(f"\n\u26a0\ufe0f  Only got {total_tokens:,} tokens ({100 * total_tokens / TARGET_TOKENS:.0f}% of budget).")
    print(f"   Consider: lowering MIN_TAMIL_PCT to 60%, or increasing MAX_DOC_CHARS.")


📥 Streaming Sangraha verified/tam (verified)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

   ...verified: 2,000 docs, 4,004,806 tokens (total: 8% of budget)
   ...verified: 4,000 docs, 7,952,946 tokens (total: 16% of budget)
   ...verified: 6,000 docs, 11,904,190 tokens (total: 24% of budget)
   ...verified: 8,000 docs, 16,042,805 tokens (total: 32% of budget)
   ...verified: 10,000 docs, 20,021,447 tokens (total: 40% of budget)
   ...verified: 12,000 docs, 24,215,740 tokens (total: 48% of budget)
   ...verified: 14,000 docs, 28,282,092 tokens (total: 57% of budget)
   ...verified: 16,000 docs, 32,261,804 tokens (total: 65% of budget)
   ...verified: 18,000 docs, 36,353,160 tokens (total: 73% of budget)
   ...verified: 20,000 docs, 40,400,821 tokens (total: 81% of budget)
   ...verified: 22,000 docs, 44,508,972 tokens (total: 89% of budget)
   ...verified: 24,000 docs, 48,638,488 tokens (total: 97% of budget)
   ...verified: 26,000 docs, 52,714,737 tokens (total: 105% of budget)
   ✅ verified: kept 27,105 / scanned 29,250 (55,002,009 tokens)

📊 FILTERING SUMMARY
Source     

## Quality Verification

Verify the cleaned corpus before packing.

In [7]:
# Verify data quality on random sample
sample_size = min(500, len(clean_texts))
sample_indices = random.sample(range(len(clean_texts)), sample_size)
sample_docs = [clean_texts[i] for i in sample_indices]

tamil_pcts = [tamil_char_pct(d) for d in sample_docs]
doc_lengths = [len(d) for d in sample_docs]
has_replacement = sum(1 for d in sample_docs if '\ufffd' in d)
has_zwc = sum(1 for d in sample_docs if any(c in d for c in '\u200b\u200c\u200d\ufeff'))

print(f"\U0001f50d Quality Check ({sample_size} random docs):")
print(f"   Tamil%: min={min(tamil_pcts):.0f}%, median={np.median(tamil_pcts):.0f}%, max={max(tamil_pcts):.0f}%")
print(f"   Length: min={min(doc_lengths)}, median={np.median(doc_lengths):.0f}, max={max(doc_lengths)}")
print(f"   Contains \\ufffd: {has_replacement}/{sample_size} {'\u274c FIX CLEANING!' if has_replacement else '\u2705'}")
print(f"   Contains zero-width: {has_zwc}/{sample_size} {'\u274c FIX CLEANING!' if has_zwc else '\u2705'}")

assert has_replacement == 0, "Replacement chars found in cleaned data!"
assert min(tamil_pcts) >= MIN_TAMIL_PCT - 1, f"Tamil% below threshold: {min(tamil_pcts)}%"
print(f"\n\u2705 Quality verification passed")

# Show a sample
print(f"\n\U0001f4d6 Sample cleaned doc:")
print(f"   Tamil%: {tamil_pcts[0]:.0f}%")
print(f"   Text:   {sample_docs[0][:300]}...")

🔍 Quality Check (500 random docs):
   Tamil%: min=71%, median=86%, max=91%
   Length: min=253, median=1508, max=7945
   Contains \ufffd: 0/500 ✅
   Contains zero-width: 0/500 ✅

✅ Quality verification passed

📖 Sample cleaned doc:
   Tamil%: 87%
   Text:   வங்கதேச கிரிக்கெட் அணி, நியூசிலாந்தில் சுற்றுப்பயணம் மேற்கொண்டு விளையாடி வருகிறது. கிறிஸ்ட் சர்ச் நகரில் நாளை 3-வது டெஸ்ட் போட்டி தொடங்க உள்ளது. அதனால், வீரர்கள் அனைவரும் மசூதிக்கு அருகே இருக்கும் நட்சத்திர ஓட்டலில் தங்க வைக்கப்பட்டனர்.
இந்நிலையில், தொழுகைக்காக வீரர்கள் அனைவரும் சொகுசுப் பேருந்து ஒன...


## Pack Into 1024-Token Blocks

In [8]:
print(f"\U0001f4e6 Packing {len(clean_texts):,} docs into {MAX_SEQ_LENGTH}-token blocks...")

all_token_ids = []
eos_id = tokenizer.eos_token_id

for i, text in enumerate(clean_texts):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    all_token_ids.extend(tokens)
    all_token_ids.append(eos_id)

    if (i + 1) % 5000 == 0:
        print(f"   ...tokenized {i + 1:,}/{len(clean_texts):,} docs")

print(f"   Total token stream: {len(all_token_ids):,} tokens")

# Split into fixed-length blocks
n_blocks = len(all_token_ids) // MAX_SEQ_LENGTH
trimmed = all_token_ids[:n_blocks * MAX_SEQ_LENGTH]
blocks = [trimmed[i * MAX_SEQ_LENGTH:(i + 1) * MAX_SEQ_LENGTH] for i in range(n_blocks)]

print(f"\n\u2705 Packed into {len(blocks):,} blocks of {MAX_SEQ_LENGTH} tokens")
print(f"   Total training tokens: {len(blocks) * MAX_SEQ_LENGTH:,}")
print(f"   Discarded tail: {len(all_token_ids) - len(trimmed):,} tokens")

# Verify sample blocks for quality
print(f"\n\U0001f50d Verifying sample blocks...")
for idx in [0, len(blocks)//2, len(blocks)-1]:
    decoded = tokenizer.decode(blocks[idx][:100])
    t_pct = tamil_char_pct(decoded)
    has_repl = '\ufffd' in decoded
    # Note: boundary artifacts may cause 1-2 replacement chars at block start
    # This is expected with byte-level BPE packing and affects <0.1% of tokens
    print(f"   Block {idx}: Tamil={t_pct:.0f}%, has_\ufffd={'yes (boundary)' if has_repl else 'no'}, text={decoded[:80]}...")

📦 Packing 27,105 docs into 1024-token blocks...
   ...tokenized 5,000/27,105 docs
   ...tokenized 10,000/27,105 docs
   ...tokenized 15,000/27,105 docs
   ...tokenized 20,000/27,105 docs
   ...tokenized 25,000/27,105 docs
   Total token stream: 55,029,114 tokens

✅ Packed into 53,739 blocks of 1024 tokens
   Total training tokens: 55,028,736
   Discarded tail: 378 tokens

🔍 Verifying sample blocks...
   Block 0: Tamil=87%, has_�=no, text=புதுச்சேரி-நேருக்கு நேர் விவாதம் செய்தால் மீசை அல்ல, மொட்டையே அடித்துக் கொள்ள நே...
   Block 26869: Tamil=69%, has_�=yes (boundary), text= 223. 01 புள்ளிகள் வீழ்ச்சியடைந்து 62,625. 63 ஆக இருந்தது. அதேநேரத்தில் தேசிய பங...
   Block 53738: Tamil=86%, has_�=no, text=் காரணத்தையும் அவற்றை உடனடியாக டெலீட் செய்ய என்ன வழி என்று பார்ப்போம்.
இதற்கு கா...


## Create Dataset & Upload to HuggingFace

In [9]:
packed_dataset = Dataset.from_dict({
    "input_ids": blocks,
    "attention_mask": [[1] * MAX_SEQ_LENGTH for _ in blocks],
    "labels": [list(b) for b in blocks],
})

split = packed_dataset.train_test_split(test_size=EVAL_PCT, seed=RANDOM_SEED)

dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

print(f"\U0001f4ca Dataset created:")
print(f"   Train:      {len(dataset_dict['train']):,} blocks ({len(dataset_dict['train']) * MAX_SEQ_LENGTH:,} tokens)")
print(f"   Validation: {len(dataset_dict['validation']):,} blocks ({len(dataset_dict['validation']) * MAX_SEQ_LENGTH:,} tokens)")

📊 Dataset created:
   Train:      52,664 blocks (53,927,936 tokens)
   Validation: 1,075 blocks (1,100,800 tokens)


In [10]:
print(f"\U0001f4e4 Uploading to {OUTPUT_DATASET}...")

# Build source summary for commit message
src_summary = ", ".join(f"{k}:{v['kept']}" for k, v in source_stats.items())

dataset_dict.push_to_hub(
    OUTPUT_DATASET,
    private=False,
    commit_message=(
        f"DAPT data v1.1: {len(blocks):,} blocks x {MAX_SEQ_LENGTH} tokens | "
        f"Sources: {src_summary} | "
        f"Filters: NFKC + Tamil>={MIN_TAMIL_PCT}% + dedup + clean | "
        f"Tokenizer: {BASE_MODEL}"
    ),
)

print(f"\n\u2705 Dataset uploaded: https://huggingface.co/datasets/{OUTPUT_DATASET}")

📤 Uploading to CryptoYogi/vazhi-dapt-tamil-v1_1...


Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  86%|########5 | 1.95MB / 2.27MB            


✅ Dataset uploaded: https://huggingface.co/datasets/CryptoYogi/vazhi-dapt-tamil-v1_1


In [11]:
# Verify upload
print(f"\U0001f50d Verifying upload...")
verify_ds = load_dataset(OUTPUT_DATASET)

print(f"   Train:      {len(verify_ds['train']):,} blocks")
print(f"   Validation: {len(verify_ds['validation']):,} blocks")

sample = verify_ds["train"][0]
assert len(sample["input_ids"]) == MAX_SEQ_LENGTH
assert sample["input_ids"] == sample["labels"]

print(f"\n\u2705 Verification passed!")

print(f"\n{'='*60}")
print(f"\U0001f4cb SUMMARY")
print(f"{'='*60}")
print(f"   Dataset:     {OUTPUT_DATASET}")
print(f"   Sources:     {[s['label'] for s in SANGRAHA_SOURCES]}")
print(f"   Cleaning:    NFKC + strip \\ufffd/ZWC/control + collapse whitespace")
print(f"   Tamil >= {MIN_TAMIL_PCT}% (raised from v1.0's 50%)")
print(f"   Total docs:  {len(clean_texts):,}")
print(f"   Total tokens: {total_tokens:,}")
print(f"   Blocks:      {len(blocks):,} x {MAX_SEQ_LENGTH}")
print(f"\n\U0001f449 Next: Run Vazhi_DAPT_v1_1_Tamil.ipynb on Kaggle T4")
print(f'   It will load: ds = load_dataset("{OUTPUT_DATASET}")')

🔍 Verifying upload...


README.md:   0%|          | 0.00/475 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/53.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/53.2M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.27M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52664 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1075 [00:00<?, ? examples/s]

   Train:      52,664 blocks
   Validation: 1,075 blocks

✅ Verification passed!

📋 SUMMARY
   Dataset:     CryptoYogi/vazhi-dapt-tamil-v1_1
   Sources:     ['verified', 'unverified', 'synthetic']
   Cleaning:    NFKC + strip \ufffd/ZWC/control + collapse whitespace
   Tamil >= 70% (raised from v1.0's 50%)
   Total docs:  27,105
   Total tokens: 55,002,009
   Blocks:      53,739 x 1024

👉 Next: Run Vazhi_DAPT_v1_1_Tamil.ipynb on Kaggle T4
   It will load: ds = load_dataset("CryptoYogi/vazhi-dapt-tamil-v1_1")
